In [19]:
from typing import Annotated, Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from typing import TypedDict
from langgraph.graph import START, END, StateGraph
from langchain_groq import ChatGroq
from operator import add

class llmResponse(BaseModel):
    feedback:str=Field(description="This is the feedback for the essay")
    score:int=Field(description="This is the score for the essay",le=10,ge=0)

essay1="""

Artificial Intelligence and the Future of Human Agency 

Artificial Intelligence, or AI, is becoming very popular today. It is used in phones, computers, and cars. AI means machines can think and work like humans. This is good in many ways. It helps us save time and do work fast. For example, doctors can use AI to find diseases early. Students can use it to learn new things.However, AI also has bad points. Many people are scared that robots will take their jobs. If machines do all the work, what will humans do? Also, we might become too lazy. We will stop using our own brains to think or solve problems. If we depend on computers for everything, we might lose our creativity and skills.In conclusion, AI is a useful tool, but we must use it carefully. We should not let machines control our lives. Humans must always remain the master, and AI should only be a helper. By doing this, we can enjoy the benefits of technology without losing our human qualities.

 """

essay2=""" 
Artificial Intelligence and the Future of Human Agency

We are drowning in information, while starving for wisdom." This remark by E.O. Wilson captures the paradox of the twenty-first century. Artificial Intelligence (AI) has transitioned from the realm of science fiction into the bedrock of modern civilization. Algorithms now curate our realities, diagnose pathologies, and forecast macroeconomic trends. Yet, beneath the veneer of unprecedented efficiency lies a profound existential friction: the steady erosion of human agency. Agency—the capacity of actors to act in a world and shape their own destinies—is the cornerstone of democratic thought and human dignity. As machine intelligence scales new cognitive frontiers, we are compelled to ask whether AI is an amplifier of human potential or an insidious architect of human obsolescence.

"""

# llm=ChatOpenAI(model="gpt-4o-mini")
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)
structured_llm=llm.with_structured_output(llmResponse)


class EssayEvalState(TypedDict):
    essay:str
    cot_feedback:str
    doa_feedback:str
    lang_feedback:str
    final_feedback:str
    scores:Annotated[list[int],add]
    avg_score:float

def cot(state:EssayEvalState)->dict:
    result=structured_llm.invoke(state["essay"])
    
    return {
        "cot_feedback":result.feedback,
        "scores":[result.score]
    }


def doa(state:EssayEvalState)->dict:
    result=structured_llm.invoke(state["essay"])
    
    return {
        "doa_feedback":result.feedback,
        "scores":[result.score]
    }


def language(state:EssayEvalState)->dict:
    result=structured_llm.invoke(state["essay"])
    
    return {
        "lang_feedback":result.feedback,
        "scores":[result.score]
    }

def finalEval(state:EssayEvalState)->dict:
    prompt=f"generate a final feedback for the essay considering {state['cot_feedback']}  {state['doa_feedback']} {state['lang_feedback']}"
    result=structured_llm.invoke(prompt)
    avgScore=sum(state['scores'])/len(state['scores'])
    return {
        "final_feedback":result.feedback,
        "avg_score":round(avgScore,2)
    }



graph = StateGraph(EssayEvalState)


graph.add_node('cot',cot)
graph.add_node('doa',doa)
graph.add_node('language',language)
graph.add_node('finalEval',finalEval)

graph.add_edge(START,'cot')
graph.add_edge(START,'doa')
graph.add_edge(START,'language')

graph.add_edge('cot','finalEval')
graph.add_edge('doa','finalEval')
graph.add_edge('language','finalEval')

graph.add_edge('finalEval',END)

workflow=graph.compile()

initial_state={
    "essay":essay2
}

final_state=workflow.invoke(initial_state)

filtered_state = {k: v for k, v in final_state.items() if k != "essay"}

print(filtered_state)


{'cot_feedback': 'The essay explores the impact of Artificial Intelligence on human agency, highlighting the tension between the benefits of AI and the potential erosion of human autonomy. The writer effectively uses a quote from E.O. Wilson to frame the discussion and raises important questions about the role of AI in shaping human destiny. The text demonstrates a strong command of language and a clear, well-structured argument. However, it could benefit from more concrete examples and evidence to support the claims made about the effects of AI on human agency.', 'doa_feedback': 'The essay explores the impact of Artificial Intelligence on human agency, highlighting the tension between the benefits of AI and the potential erosion of human autonomy. The writer effectively uses a quote from E.O. Wilson to frame the discussion and raises important questions about the role of AI in shaping human destiny. The text demonstrates a strong command of language and a clear, well-structured argume

In [ ]:
{'cot_feedback': "This essay introduction sets a thought-provoking tone by juxtaposing the advancements in AI with the implications for human agency. The quote from E.O. Wilson effectively frames the discussion, highlighting the dilemma faced in the age of information. However, expanding on what is meant by 'human agency' and giving brief examples of how AI affects specific aspects of life (like personal decision-making, employment, etc.) could enhance clarity and engagement. Additionally, it would be beneficial to introduce a thesis statement that outlines your argument or perspective on whether AI is ultimately beneficial or detrimental to human agency. Overall, it’s a compelling start that requires a clearer argumentative structure to fully develop the ideas presented.", 'doa_feedback': "This essay presents a thought-provoking analysis of the intersection between artificial intelligence and human agency. It effectively utilizes a relevant quote to introduce the main theme and highlights the complexities of our reliance on AI. The language is articulate, and the choice of vocabulary enhances the intellectual tone. However, it would benefit from a clearer structure with defined sections that explore specific aspects of the topic, including definitions of key terms, examples of AI applications, and discussions on the philosophical implications. Additionally, contrasting viewpoints on AI's role in society could enrich the essay further.", 'lang_feedback': "This introduction effectively sets the stage for a critical discussion on the impact of artificial intelligence on human agency. It poses important questions about the dual nature of AI — as both a beneficial tool and a potential threat to human autonomy. The inclusion of a quote from E.O. Wilson adds depth and establishes the urgency of the topic. To strengthen the essay further, consider providing specific examples of AI's influence on human decision-making, as well as potential solutions to preserve human agency in the face of increasing automation.", 'final_feedback': "This essay introduction effectively raises critical questions surrounding the relationship between artificial intelligence and human agency. The juxtaposition of AI advancements and their implications captures the reader's interest, particularly with the strong quote from E.O. Wilson. However, the essay could greatly benefit from a clearer argumentative structure. First, expounding on the concept of 'human agency' would enhance understanding; consider defining it explicitly and providing examples of how AI influences personal choices, employment, and daily life. A clear thesis statement outlining your stance—whether AI is a force for good or ill in relation to human agency—would give the essay a more defined path. The use of articulate language and sophisticated vocabulary sets a significant intellectual tone, but organizing the essay into distinct sections dedicated to definitions, specific AI applications, and philosophical discussions will aid in clarity. Additionally, incorporating contrasting viewpoints on AI's societal role will provide a well-rounded perspective. Finally, the suggestion of including specific examples of AI's impact on decision-making, along with solutions to safeguard human agency against over-reliance on technology, would enhance the depth of the argument. Overall, this is a compelling introduction with the potential for a deeper, more engaging exploration of its themes.", 'scores': [7, 7, 8], 'avg_score': 7.33}

hi
